##Notebook de Entrada

Notebook de entrada principal do algoritmo de neuroimunologia, os dados são extraidos das seguintes tabelas:

gold_corporativo_ia.corporativo.tb_gold_mov_exame

gold_corporativo_ia.corporativo.tb_gold_mov_paciente

Após realizar o filtro dos exames os dados são enriquecidos com as informações dos pacientes, posterior a view resultante é usada para inserção dos dados na tabela final.

### Variáveis de Ambiente

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")

In [0]:
environment = dbutils.widgets.get("environment")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

In [0]:
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_neuroimunologia"
VW_BALIZADORA = f"{environment_tbl}vw_neuroimunologia_balizador"
SCHEMA = "ia"

In [0]:
print('Environment:   ', environment_tbl)
print('Tabela destino:', OUTPUT_TABLENAME)
print('VW balizadora: ', VW_BALIZADORA)
print('Schema:        ', SCHEMA)

In [0]:
# spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{OUTPUT_TABLENAME}""")

### Tabelas & Views

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
WITH base AS (
  SELECT
      e.id_unidade                                     AS idunidade,
      e.nme_hospital                                   AS unidade,
      p.id_paciente                                    AS id_pct,
      p.cli_nome                                       AS paciente,
      p.cli_genero                                     AS sexo,
      p.cli_data_nascimento                            AS nascimento,
      p.doc_cpf                                        AS cpf,
      p.cli_telefone_numero                            AS telefonePaciente,
      p.cli_telefone_ddd                               AS telefonePacienteDDD,
      e.tp_procedimento                                AS procedimento,
      e.num_pedido_integracao                          AS num_pedido_integracao,
      e.nme_regional_hospital                          AS nme_regional_hospital,
      e.nme_convenio                                   AS nme_convenio,

      regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), e.proced_descricao_ajustado, '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,
      regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
      regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

      /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
      FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
      CASE
        WHEN exame_nr RLIKE '(rm|ressonancia|resonancia)' 
        THEN 'RM'
        ELSE 'IND'
      END                                              AS modalidade,
      
      translate(
        regexp_replace(exame_nr, '[\\u00A0\\u2007\\u202F]', ' '),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS exame,

      e.dt_dia_exame                                   AS dataexame,
      translate(
        regexp_replace(
          lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
          '[\\u00A0\\u2007\\u202F]', ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS tipoexame,

      CAST(e.id_exame AS STRING)                       AS an,

      e.dt_liberacao_laudo                             AS DataLaudoLiberado,
      array_join(
        transform(e.proced_lista_exames, x -> x.laudo_transformado),
        '\n'
      ) AS Laudo,
      COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
      CASE
        WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
        WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
        ELSE 0
      END AS idstatus

  FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
  LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
         ON p.id_paciente = e.id_patient
),
filtrada AS (
  SELECT *
  FROM base
  WHERE
    UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
    
  AND (LOWER(exame) LIKE 'rm%' AND LOWER(exame) RLIKE 'cranio | orbitas | cervical | toracica')
    AND LOWER(exame) NOT RLIKE 'desfiladeiro|parede toracica|braquial|radioterapia'
 
  AND exame != ""
  AND DataLaudoFinal IS NOT NULL
  AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 150)
                                 AND date_sub(current_date(), 1) -- ontem
),

dedup AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (
      PARTITION BY an
      ORDER BY TIMESTAMP(DataLaudoFinal) DESC
    ) AS rn,
    CASE WHEN ROW_NUMBER() OVER (
             PARTITION BY an
             ORDER BY TIMESTAMP(DataLaudoFinal) DESC
         ) = 1
         THEN 1 ELSE 0 END AS an_duplicado
  FROM filtrada f
)
SELECT
  idunidade,
  unidade,
  id_pct,
  paciente,
  telefonePacienteDDD,
  telefonePaciente,
  sexo,
  nascimento,
  idade_anos,
  cpf,
  modalidade,
  exame,
  num_pedido_integracao,
  nme_regional_hospital,
  nme_convenio,
  dataexame,
  tipoexame,
  an,
  an_duplicado,
  idstatus,
  -- 4 AS idstatus,                 -- mantido
  -- idLaudo,                       
  DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
  Laudo,
  -- current_timestamp() as datCarga
  CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
FROM dedup;
""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{OUTPUT_TABLENAME}
            USING DELTA
            AS
            SELECT *
            FROM {VW_BALIZADORA}
            WHERE 1 = 0
            """)

In [0]:
# spark.sql(f"""
# ALTER TABLE {SCHEMA}.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

In [0]:
spark.sql(f"""
INSERT INTO {SCHEMA}.{OUTPUT_TABLENAME} (
    idunidade,
    unidade,
    id_pct,
    paciente,
    telefonePacienteDDD,
    telefonePaciente,
    sexo,
    nascimento,
    idade_anos,
    cpf,
    modalidade,
    exame,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    dataexame,
    tipoexame,
    an,
    an_duplicado,
    idstatus,
    DataLaudoLiberado,
    Laudo,
    dataExecucaoModelo
)
SELECT
    s.idunidade,
    s.unidade,
    s.id_pct,
    s.paciente,
    s.telefonePacienteDDD,
    s.telefonePaciente,
    s.sexo,
    s.nascimento,
    s.idade_anos,
    s.cpf,
    s.modalidade,
    s.exame,
    s.num_pedido_integracao,
    s.nme_regional_hospital,
    s.nme_convenio,
    s.dataexame,              
    s.tipoexame,
    s.an,
    s.an_duplicado,
    s.idstatus,
    s.DataLaudoLiberado,
    s.Laudo,
    s.dataExecucaoModelo
FROM {VW_BALIZADORA} s
LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
    ON t.an        = s.an
""")

In [0]:
# spark.sql(f"""INSERT INTO {CATALOG}.{OUTPUT_TABLENAME}
#               SELECT s.*
#               FROM {VW_BALIZADORA} s
#               LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
#                 ON  t.idunidade = s.idunidade
#                 AND t.id_pct = s.id_pct
#                 AND t.exame = s.exame
#                 AND t.dataexame = s.dataexame
#                 AND t.tipoexame = s.tipoexame
#                 AND t.an = s.an
#             """)

### Consultas

In [0]:
spark.sql(f"""
            select *
            from {SCHEMA}.{OUTPUT_TABLENAME}
            where cast(dataExecucaoModelo as date) = current_date()
            """).display()

In [0]:
spark.sql(f"""
            SELECT
                dataExecucaoModelo,
                COUNT(*) AS total_registros
            FROM {SCHEMA}.{OUTPUT_TABLENAME}
            GROUP BY dataExecucaoModelo
            ORDER BY dataExecucaoModelo
        """).display()

In [0]:
spark.sql(f"""
SELECT
  CAST(DataLaudoLiberado AS DATE) AS data_exame,
  unidade,
  COUNT(*) AS total_exames
FROM {SCHEMA}.{OUTPUT_TABLENAME}
GROUP BY
  CAST(DataLaudoLiberado AS DATE),
  unidade
ORDER BY
  data_exame DESC,
  unidade
""").display()

In [0]:
dbutils.notebook.exit("Final Notebook")